[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C68_Eval_Infrastructure_Course/03_result_store/03_result_store.ipynb)

# 03 · 结果存储与分析（宽事实表 / 后置聚合 / 切片与 Simpson / run diff / 完整性校验）

目标：把结果层做成「决定你未来能问出什么问题」的那一层。

本 notebook 你会亲手实现：
1. **宽事实表与只追加写入** —— 11 个字段的最小 schema
2. **后置聚合** —— 同一批结果算出 micro / macro / pass^k / 加权四个数
3. **切片与 Simpson 悖论** —— 复现「每一片都更好，总分却更低」
4. **交集重算** —— 任务集不一致时唯一正确的比较方式
5. **run diff** —— fixed / regressed / churn + McNemar，把「涨了 3 点」变成可行动的信息
6. **完整性校验** —— 五条应当写进加载函数的断言

> 心智模型：**存最细粒度的事实，把聚合留到查询时做。
> 你现在想不到的问题，半年后一定会被问到——而那时明细还在，就是一句查询。**

## 0 · 环境与结果表

In [ ]:
import os, json, math, sqlite3, shutil, random, itertools
from collections import Counter, defaultdict

import numpy as np

TMP = os.path.abspath('./_eval_tmp')
if os.path.exists(TMP):
    shutil.rmtree(TMP)
os.makedirs(TMP, exist_ok=True)

SCHEMA = '''CREATE TABLE IF NOT EXISTS results (
    run_id       TEXT,
    task_id      TEXT,
    attempt      INTEGER,
    fingerprint  TEXT,
    dataset_ver  TEXT,
    scorer_ver   TEXT,
    status       TEXT,
    score        REAL,
    output       TEXT,
    cost_usd     REAL,
    latency_ms   INTEGER,
    PRIMARY KEY (run_id, task_id, attempt))'''

VALID_STATUS = {'ok', 'timeout', 'refusal', 'parse_error', 'scorer_error'}

class ResultStore:
    def __init__(self, path):
        self.conn = sqlite3.connect(path)
        self.conn.execute(SCHEMA)
        self.conn.commit()

    def put_many(self, rows):
        """只追加、幂等（主键冲突则忽略）。返回真正写入的行数。"""
        before = self.count()
        self.conn.executemany(
            'INSERT OR IGNORE INTO results VALUES (:run_id,:task_id,:attempt,:fingerprint,'
            ':dataset_ver,:scorer_ver,:status,:score,:output,:cost_usd,:latency_ms)', rows)
        self.conn.commit()
        return self.count() - before

    def count(self):
        return self.conn.execute('SELECT COUNT(*) FROM results').fetchone()[0]

    def rows(self, run_id=None):
        q = 'SELECT * FROM results' + (' WHERE run_id=?' if run_id else '')
        cur = self.conn.execute(q, (run_id,) if run_id else ())
        cols = [d[0] for d in cur.description]
        return [dict(zip(cols, r)) for r in cur.fetchall()]

store = ResultStore(os.path.join(TMP, 'results.db'))
print('结果表就位，字段:', [c.split()[0] for c in SCHEMA.split('(')[1].split(',')][:11])
print('\n✅ 主键是 (run_id, task_id, attempt) —— 幂等性、pass^k、逐题 diff 全靠它。')

## 1 · 造一批带任务属性的结果

任务属性（难度、子系统、判分器强度）**不存在结果表里**，而是来自数据集，
分析时 join 上去 —— 这样任务属性被重新标注时，历史结果不需要迁移。

In [ ]:
rng = np.random.default_rng(0)
SUBSYSTEMS = ['billing', 'flight', 'account']
DIFFICULTIES = ['easy', 'medium', 'hard']

TASK_META = {}
for i in range(500):
    tid = f't{i:03d}'
    TASK_META[tid] = {
        'task_id': tid,
        'difficulty': DIFFICULTIES[0] if i < 250 else (DIFFICULTIES[1] if i < 420 else DIFFICULTIES[2]),
        'subsystem': SUBSYSTEMS[i % 3],
        # 判分器强度（C66-02 的变异分数）：低于 0.6 的任务判分不可信
        'scorer_strength': float(np.clip(rng.beta(5, 2), 0, 1)),
    }

BASE_ACC = {'easy': 0.88, 'medium': 0.55, 'hard': 0.22}

def synth_run(run_id, fingerprint, attempts=3, acc_shift=0.0, seed=0,
              task_ids=None, dataset_ver='v3', scorer_ver='v2',
              p_timeout=0.02, p_scorer_err=0.004):
    r = np.random.default_rng(seed)
    ids = task_ids if task_ids is not None else list(TASK_META)
    out = []
    for tid in ids:
        m = TASK_META[tid]
        p = float(np.clip(BASE_ACC[m['difficulty']] + acc_shift, 0.01, 0.99))
        for a in range(attempts):
            u = r.random()
            if u < p_scorer_err:
                st, sc = 'scorer_error', None
            elif u < p_scorer_err + p_timeout:
                st, sc = 'timeout', None
            else:
                st, sc = 'ok', float(r.random() < p)
            out.append({'run_id': run_id, 'task_id': tid, 'attempt': a,
                        'fingerprint': fingerprint, 'dataset_ver': dataset_ver,
                        'scorer_ver': scorer_ver, 'status': st, 'score': sc,
                        'output': None if st == 'ok' else f'<{st} raw output>',
                        'cost_usd': float(r.gamma(2, 0.004)),
                        'latency_ms': int(r.gamma(3, 900))})
    return out

ROWS_A = synth_run('run-A', 'fpA00001', seed=1, acc_shift=0.0)
ROWS_B = synth_run('run-B', 'fpB00002', seed=2, acc_shift=0.05)
n_w = store.put_many(ROWS_A) + store.put_many(ROWS_B)
print(f'写入 {n_w} 行 | 表内共 {store.count()} 行')
assert n_w == len(ROWS_A) + len(ROWS_B)
# 幂等：重写一遍不产生新行
assert store.put_many(ROWS_A) == 0
print('重复写入新增行数: 0  ✓ 幂等')
print('\n✅ 任务属性存在 TASK_META 里而不是结果表里——')
print('   难度被重新标注时（模块 01 的 meta_changed），历史结果一行都不用改。')

## 2 · 后置聚合：同一批结果，四个都「对」的数字

In [ ]:
def _usable(rows):
    """分母：排除 scorer_error（模块 02 的唯一可排除项），其余全部计入。"""
    return [r for r in rows if r['status'] != 'scorer_error']

def agg_micro(rows):
    u = _usable(rows)
    vals = [(r['score'] or 0.0) for r in u]
    return {'value': (sum(vals) / len(u)) if u else float('nan'),
            'n_rows': len(u), 'n_excluded': len(rows) - len(u), 'unit': 'rollout'}

def agg_macro_task(rows):
    by = defaultdict(list)
    for r in _usable(rows):
        by[r['task_id']].append(r['score'] or 0.0)
    means = [np.mean(v) for v in by.values() if v]
    return {'value': float(np.mean(means)) if means else float('nan'),
            'n_rows': len(means), 'n_excluded': len(rows) - len(_usable(rows)),
            'unit': 'task'}

def agg_macro_slice(rows, key):
    by = defaultdict(list)
    for r in _usable(rows):
        by[TASK_META[r['task_id']][key]].append(r['score'] or 0.0)
    means = [np.mean(v) for v in by.values() if v]
    return {'value': float(np.mean(means)) if means else float('nan'),
            'n_rows': len(means), 'n_excluded': len(rows) - len(_usable(rows)),
            'unit': key}

def agg_pass_pow_k(rows, k=3):
    by = defaultdict(list)
    for r in _usable(rows):
        by[r['task_id']].append(r['score'] or 0.0)
    hits = [1.0 if (len(v) >= k and all(x == 1.0 for x in v[:k])) else 0.0
            for v in by.values()]
    return {'value': float(np.mean(hits)) if hits else float('nan'),
            'n_rows': len(hits), 'n_excluded': len(rows) - len(_usable(rows)),
            'unit': f'task (pass^{k})'}

rows_a = store.rows('run-A')
print(f"{'口径':<26}{'数值':>10}{'分母':>8}{'排除':>8}{'单位':>16}")
for name, fn in [('micro（全部 rollout）', agg_micro),
                 ('macro by task', agg_macro_task),
                 ('macro by difficulty', lambda r: agg_macro_slice(r, 'difficulty')),
                 ('pass^3', agg_pass_pow_k)]:
    a = fn(rows_a)
    print(f'{name:<26}{a["value"]:>10.1%}{a["n_rows"]:>8}{a["n_excluded"]:>8}{a["unit"]:>16}')

m = agg_micro(rows_a); md_ = agg_macro_slice(rows_a, 'difficulty'); pk = agg_pass_pow_k(rows_a)
assert md_['value'] < m['value'], 'easy 题多，micro 被它们抬高；macro 等权后下降'
assert pk['value'] < m['value'], 'pass^3 必然低于单次成功率'
assert all(a['n_excluded'] > 0 for a in [m, md_, pk])
print('\n✅ 四个数字都是「对」的，但回答的是四个不同的问题。')
print('   **只有存了明细，换口径才是一句查询而不是一次重跑。**')
print('   注意每个数字都带着分母与排除数——这是最便宜的可信度提升。')

In [ ]:
# 空输入必须返回 nan 而不是 0
empty = agg_micro([])
assert math.isnan(empty['value']), '分母为 0 必须返回 nan'
all_err = agg_micro([{'status': 'scorer_error', 'score': None, 'task_id': 't000'}] * 5)
assert math.isnan(all_err['value']) and all_err['n_excluded'] == 5
print('空输入 → nan ✓ | 全是判分器崩溃 → nan，排除 5 行 ✓')
print('✅ 返回 0 会让「没有数据」和「全错」在下游变得无法区分——')
print('   而这两者在告警逻辑里应当触发完全不同的动作。')

## 3 · 切片与 Simpson 悖论：每一片都更好，总分却更低

In [ ]:
def slice_table(rows, key):
    by = defaultdict(list)
    for r in _usable(rows):
        by[TASK_META[r['task_id']][key]].append(r['score'] or 0.0)
    return {k: (float(np.mean(v)), len(v)) for k, v in sorted(by.items())}

# 构造 Simpson：模型 X 每一片都更好，但它的任务构成偏向难题
EASY_IDS = [t for t, m in TASK_META.items() if m['difficulty'] == 'easy']
HARD_IDS = [t for t, m in TASK_META.items() if m['difficulty'] == 'hard']

rows_X = (synth_run('run-X', 'fpX', attempts=3, seed=11, acc_shift=0.12,
                    task_ids=EASY_IDS[:60], p_timeout=0.0, p_scorer_err=0.0)
          + synth_run('run-X', 'fpX', attempts=3, seed=12, acc_shift=0.12,
                      task_ids=HARD_IDS, p_timeout=0.0, p_scorer_err=0.0))
rows_Y = (synth_run('run-Y', 'fpY', attempts=3, seed=13, acc_shift=0.0,
                    task_ids=EASY_IDS, p_timeout=0.0, p_scorer_err=0.0)
          + synth_run('run-Y', 'fpY', attempts=3, seed=14, acc_shift=0.0,
                      task_ids=HARD_IDS[:30], p_timeout=0.0, p_scorer_err=0.0))

sx, sy = slice_table(rows_X, 'difficulty'), slice_table(rows_Y, 'difficulty')
print(f"{'难度':<10}{'模型X':>16}{'模型Y':>16}")
for d in ['easy', 'hard']:
    vx, nx = sx.get(d, (float('nan'), 0)); vy, ny = sy.get(d, (float('nan'), 0))
    print(f'{d:<10}{f"{vx:.0%} (n={nx})":>16}{f"{vy:.0%} (n={ny})":>16}')
tx, ty = agg_micro(rows_X)['value'], agg_micro(rows_Y)['value']
print(f'{"总计":<10}{tx:>16.0%}{ty:>16.0%}')

assert sx['easy'][0] > sy['easy'][0] and sx['hard'][0] > sy['hard'][0], 'X 每一片都更好'
assert tx < ty, '但 X 的总分更低'
print('\n⚠️ Simpson 悖论：X 在每个难度上都更好，总分却更低——')
print('   因为 X 跑的难题占比更高。**总分被「跑了哪些题」这个混杂变量污染了。**')
print('   在评测里这最常见的成因不是有人故意，而是两次运行的任务集不完全相同')
print('   （超时被排除、新加了题、某些题在一次运行里失败被跳过）。')

In [ ]:
def intersect_recompute(rows_a, rows_b):
    """唯一正确的比较方式：在共同任务集上重算两边（= 流行病学的直接标准化）。"""
    ids_a = {r['task_id'] for r in _usable(rows_a)}
    ids_b = {r['task_id'] for r in _usable(rows_b)}
    common = ids_a & ids_b
    fa = [r for r in rows_a if r['task_id'] in common]
    fb = [r for r in rows_b if r['task_id'] in common]
    return {'n_common': len(common), 'n_only_a': len(ids_a - common),
            'n_only_b': len(ids_b - common),
            'a': agg_micro(fa)['value'], 'b': agg_micro(fb)['value']}

ic = intersect_recompute(rows_X, rows_Y)
print(f'任务集: X 独有 {ic["n_only_a"]} 条 | 共同 {ic["n_common"]} 条 | Y 独有 {ic["n_only_b"]} 条')
print(f'朴素比较:   X {tx:.1%}  vs  Y {ty:.1%}   → 差 {tx-ty:+.1%}')
print(f'交集重算:   X {ic["a"]:.1%}  vs  Y {ic["b"]:.1%}   → 差 {ic["a"]-ic["b"]:+.1%}')
assert ic['n_common'] > 0
assert (tx - ty) < 0 < (ic['a'] - ic['b']), '交集重算把结论翻了过来'
print('\n✅ 交集重算把结论翻了过来——而这才是正确的那个。')
print('   → 报告规范：**比较两次运行前必须先检查任务集是否一致，不一致就在交集上重算。**')
print('   这条检查应当是自动的（模块 04 会把它变成 CI 里的一行断言）。')

## 4 · Run diff：从「涨了 3 点」到「哪些题变了」

In [ ]:
def task_scores(rows, agg='mean'):
    by = defaultdict(list)
    for r in _usable(rows):
        by[r['task_id']].append(r['score'] or 0.0)
    if agg == 'mean':
        return {t: float(np.mean(v)) for t, v in by.items() if v}
    return {t: float(v[0]) for t, v in by.items() if v}

def run_diff(rows_a, rows_b, thresh=0.5):
    """逐题二值化后比较。返回 fixed / regressed / churn / net + McNemar。"""
    a, b = task_scores(rows_a), task_scores(rows_b)
    common = sorted(set(a) & set(b))
    fixed = [t for t in common if a[t] < thresh <= b[t]]
    regressed = [t for t in common if b[t] < thresh <= a[t]]
    f, r = len(fixed), len(regressed)
    if f + r == 0:
        chi2, p = 0.0, 1.0
    else:
        chi2 = max(abs(f - r) - 1, 0) ** 2 / (f + r)
        p = math.erfc(math.sqrt(chi2 / 2))
    return {'n_common': len(common), 'fixed': f, 'regressed': r,
            'churn': f + r, 'net': f - r, 'chi2': chi2, 'p': p,
            'fixed_ids': fixed[:5], 'regressed_ids': regressed[:5]}

d = run_diff(store.rows('run-A'), store.rows('run-B'))
print(f'共同任务 {d["n_common"]} 条')
print(f'  修好了 (fixed)      {d["fixed"]:>4}')
print(f'  退化了 (regressed)  {d["regressed"]:>4}   ← 净提升为正也可能伴随大量退化')
print(f'  翻转总数 (churn)    {d["churn"]:>4}')
print(f'  净变化 (net)        {d["net"]:>+4}')
print(f'  McNemar chi2={d["chi2"]:.2f}  p={d["p"]:.4f}')
print(f'  退化的样例: {d["regressed_ids"]}')
assert d['churn'] >= abs(d['net'])
ratio = d['churn'] / max(abs(d['net']), 1)
print(f'\nchurn / |net| = {ratio:.1f}')
if ratio > 4:
    print('⚠️ churn 远大于净变化 → 模型在这些题上本来就不稳定，')
    print('   这次的「提升」很可能只是采样运气。**该增加重复次数，而不是宣布提升。**')
print('\n✅ 「涨了 3 点」不可行动；「修好 42 道、退化 30 道，其中 XXX 是核心路径」可行动。')

In [ ]:
# 逐题变化矩阵：按切片看退化集中在哪里
def regression_by_slice(rows_a, rows_b, key, thresh=0.5):
    a, b = task_scores(rows_a), task_scores(rows_b)
    common = set(a) & set(b)
    tab = defaultdict(lambda: {'fixed': 0, 'regressed': 0, 'n': 0})
    for t in common:
        k = TASK_META[t][key]
        tab[k]['n'] += 1
        if a[t] < thresh <= b[t]:
            tab[k]['fixed'] += 1
        elif b[t] < thresh <= a[t]:
            tab[k]['regressed'] += 1
    return dict(tab)

rb = regression_by_slice(store.rows('run-A'), store.rows('run-B'), 'difficulty')
print(f"{'难度':<10}{'n':>6}{'fixed':>8}{'regressed':>12}{'net':>8}")
for k in ['easy', 'medium', 'hard']:
    v = rb.get(k, {'n': 0, 'fixed': 0, 'regressed': 0})
    print(f'{k:<10}{v["n"]:>6}{v["fixed"]:>8}{v["regressed"]:>12}{v["fixed"]-v["regressed"]:>+8}')
assert sum(v['n'] for v in rb.values()) == d['n_common']
print('\n✅ 这张表回答的是「提升来自哪里、退化集中在哪里」——')
print('   如果退化全部集中在 easy 上，那是一个比「总分涨了」严重得多的信号。')

## 5 · 完整性校验：五条写进加载函数的断言

In [ ]:
def integrity_check(rows, expected_tasks, expected_attempts):
    problems = []
    # ① 指纹唯一
    fps = {r['fingerprint'] for r in rows}
    if len(fps) != 1:
        problems.append(f'指纹不唯一: {sorted(fps)} → run_id 被复用了')
    # ② 覆盖完整（比对**预期行数**，而不是已有行数的内部一致性）
    expected = expected_tasks * expected_attempts
    if len(rows) != expected:
        problems.append(f'行数 {len(rows)} != 预期 {expected} → 有任务被静默跳过')
    # ③ 无重复主键
    keys = [(r['run_id'], r['task_id'], r['attempt']) for r in rows]
    if len(set(keys)) != len(keys):
        problems.append('主键重复 → 幂等写入被绕过')
    # ④ 状态合法 且 status=='ok' ⟺ score 非空
    bad_status = {r['status'] for r in rows} - VALID_STATUS
    if bad_status:
        problems.append(f'非法状态: {bad_status}')
    mismatched = sum(1 for r in rows if (r['status'] == 'ok') != (r['score'] is not None))
    if mismatched:
        problems.append(f'{mismatched} 行的 status 与 score 不一致')
    # ⑤ 数据集版本一致
    dvs = {r['dataset_ver'] for r in rows}
    if len(dvs) != 1:
        problems.append(f'数据集版本不一致: {sorted(dvs)}')
    return (len(problems) == 0, problems)

ok, probs = integrity_check(store.rows('run-A'), len(TASK_META), 3)
print('run-A 完整性:', ok, probs)
assert ok

# 三种真实事故
bad1 = store.rows('run-A') + [dict(store.rows('run-B')[0], run_id='run-A')]   # 指纹混入
bad2 = store.rows('run-A')[:-30]                                              # 任务被静默跳过
bad3 = [dict(r) for r in store.rows('run-A')]; bad3[0]['status'] = 'ok'; bad3[0]['score'] = None
for name, rows_bad, n_task in [('run_id 被复用', bad1, len(TASK_META)),
                               ('任务被静默跳过', bad2, len(TASK_META)),
                               ('status 与 score 不一致', bad3, len(TASK_META))]:
    ok_, p_ = integrity_check(rows_bad, n_task, 3)
    print(f'{name:<24} 通过={ok_}  → {p_[0][:44]}')
    assert not ok_
print('\n✅ 五条检查全部生效。**它们必须放在加载函数里，而不是一个「记得跑一下」的脚本里**——')
print('   放在加载函数里，它就不可能被忘记。')

## ✏️ 练习 1：带分母的聚合函数

实现 `agg_with_denominator(rows, filter_fn=None)`：返回
`{'value', 'n_rows', 'n_excluded_scorer', 'n_filtered_out'}`。
`filter_fn(row) -> bool` 用来做切片（例如只保留判分器强度 ≥ 0.6 的任务）。
分母排除 `scorer_error`；被 `filter_fn` 过滤掉的单独计数。

In [ ]:
def agg_with_denominator(rows, filter_fn=None):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
all_rows = store.rows('run-A')
a1 = agg_with_denominator(all_rows)
assert abs(a1['value'] - agg_micro(all_rows)['value']) < 1e-12
assert a1['n_excluded_scorer'] > 0 and a1['n_filtered_out'] == 0

strong_only = agg_with_denominator(
    all_rows, filter_fn=lambda r: TASK_META[r['task_id']]['scorer_strength'] >= 0.6)
print(f"全部任务:       {a1['value']:.1%}  (n={a1['n_rows']}, 排除判分器崩溃 {a1['n_excluded_scorer']})")
print(f"只用强判分任务: {strong_only['value']:.1%}  (n={strong_only['n_rows']}, "
      f"被过滤 {strong_only['n_filtered_out']})")
assert strong_only['n_rows'] < a1['n_rows'] and strong_only['n_filtered_out'] > 0
print('✅ 练习 1 通过：「只用判分器强的任务重算一遍」是 C66-02 的敏感性分析——')
print('   有明细在，它就是一个 filter_fn，而不是一次重跑。')

## ✏️ 练习 2：任务集一致性检查

实现 `same_task_set(rows_a, rows_b)`：返回
`(是否一致, 只在 A 里的任务数, 只在 B 里的任务数)`。
任务集只统计 `status != 'scorer_error'` 的行涉及的 task_id。

In [ ]:
def same_task_set(rows_a, rows_b):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
same, oa, ob = same_task_set(store.rows('run-A'), store.rows('run-B'))
print(f'run-A vs run-B: 一致={same} (A 独有 {oa}, B 独有 {ob})')
diff_, oa2, ob2 = same_task_set(rows_X, rows_Y)
print(f'run-X vs run-Y: 一致={diff_} (X 独有 {oa2}, Y 独有 {ob2})')
assert diff_ is False and oa2 > 0 and ob2 > 0
assert same_task_set(store.rows('run-A'), store.rows('run-A'))[0] is True
print('✅ 练习 2 通过：这个函数应当在任何跨 run 比较之前被调用——')
print('   返回 False 时，直接比较总分是无意义的（Simpson），必须走交集重算。')

## ✏️ 练习 3：churn 与净变化的判读

实现 `diff_verdict(diff, min_net=10, max_churn_ratio=4.0, alpha=0.05)`：
根据 run_diff 的结果返回结论字符串：
- `p >= alpha` → `'not_significant'`
- `churn / max(|net|,1) > max_churn_ratio` → `'noisy'`（波动太大，该加重复次数）
- `net >= min_net` → `'improved'`；`net <= -min_net` → `'regressed'`
- 其余 → `'inconclusive'`

判定顺序就是上面的顺序。

In [ ]:
def diff_verdict(diff, min_net=10, max_churn_ratio=4.0, alpha=0.05):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert diff_verdict({'p': 0.4, 'churn': 10, 'net': 2}) == 'not_significant'
assert diff_verdict({'p': 0.001, 'churn': 100, 'net': 5}) == 'noisy'
assert diff_verdict({'p': 0.001, 'churn': 60, 'net': 40}) == 'improved'
assert diff_verdict({'p': 0.001, 'churn': 60, 'net': -40}) == 'regressed'
assert diff_verdict({'p': 0.001, 'churn': 12, 'net': 4}) == 'inconclusive'
print('实际 run-A → run-B 的判读:', diff_verdict(d))
print(f'  (net={d["net"]}, churn={d["churn"]}, p={d["p"]:.4f})')
print('✅ 练习 3 通过：`noisy` 这一档是最有价值的——')
print('   它把「显著但不可靠」的情况单独拎了出来，而 p 值本身区分不了这个。')

## ✏️ 练习 4：存储规模估算

实现 `storage_estimate(n_tasks, attempts, runs_per_year, bytes_per_row_core=100,
bytes_per_output=2000, keep_output_for=('timeout','refusal','parse_error','scorer_error'),
fail_rate=0.03)`：返回
`{'rows', 'core_gb', 'output_gb', 'total_gb'}`。
核心列每行 `bytes_per_row_core`；只有失败样本保留 output。

In [ ]:
def storage_estimate(n_tasks, attempts, runs_per_year,
                     bytes_per_row_core=100, bytes_per_output=2000,
                     fail_rate=0.03):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
SCENARIOS = [('CI smoke', 60, 1, 5000), ('每日全量', 500, 3, 365), ('模型选型', 500, 5, 300)]
print(f"{'场景':<14}{'行数':>12}{'核心列 GB':>12}{'output GB':>12}{'合计 GB':>10}")
total = 0.0
for name, nt, at, ry in SCENARIOS:
    e = storage_estimate(nt, at, ry)
    total += e['total_gb']
    print(f'{name:<14}{e["rows"]:>12,}{e["core_gb"]:>12.2f}{e["output_gb"]:>12.2f}'
          f'{e["total_gb"]:>10.2f}')
print(f'{"合计":<14}{"":>12}{"":>12}{"":>12}{total:>10.2f}')
e1 = storage_estimate(500, 3, 365)
assert e1['rows'] == 500 * 3 * 365
assert e1['core_gb'] < 1.0, '核心列很小'
assert e1['total_gb'] > e1['core_gb']
assert total < 20, '三个场景加起来仍然是小数据'
print('\n✅ 练习 4 通过：几百万行、十几 GB——对任何数据库都是小数据。')
print('   所以默认策略应当是**全部保留**，而不是一上来就设计复杂的归档流程。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def agg_with_denominator(rows, filter_fn=None):
    kept = [r for r in rows if (filter_fn is None or filter_fn(r))]
    n_filtered = len(rows) - len(kept)
    usable = [r for r in kept if r['status'] != 'scorer_error']
    n_excl = len(kept) - len(usable)
    vals = [(r['score'] or 0.0) for r in usable]
    return {'value': (sum(vals) / len(usable)) if usable else float('nan'),
            'n_rows': len(usable), 'n_excluded_scorer': n_excl,
            'n_filtered_out': n_filtered}

In [ ]:
# 练习 2 参考答案
def same_task_set(rows_a, rows_b):
    ia = {r['task_id'] for r in rows_a if r['status'] != 'scorer_error'}
    ib = {r['task_id'] for r in rows_b if r['status'] != 'scorer_error'}
    return (ia == ib, len(ia - ib), len(ib - ia))

In [ ]:
# 练习 3 参考答案
def diff_verdict(diff, min_net=10, max_churn_ratio=4.0, alpha=0.05):
    if diff['p'] >= alpha:
        return 'not_significant'
    if diff['churn'] / max(abs(diff['net']), 1) > max_churn_ratio:
        return 'noisy'
    if diff['net'] >= min_net:
        return 'improved'
    if diff['net'] <= -min_net:
        return 'regressed'
    return 'inconclusive'

In [ ]:
# 练习 4 参考答案
def storage_estimate(n_tasks, attempts, runs_per_year,
                     bytes_per_row_core=100, bytes_per_output=2000,
                     fail_rate=0.03):
    rows = n_tasks * attempts * runs_per_year
    core = rows * bytes_per_row_core
    out = rows * fail_rate * bytes_per_output
    GB = 1024 ** 3
    return {'rows': rows, 'core_gb': core / GB, 'output_gb': out / GB,
            'total_gb': (core + out) / GB}

---
## 🧪 真实工程胶囊：结果层的落地

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 加载函数里必须包含完整性校验（不是一个"记得跑"的脚本）
# ══════════════════════════════════════════════════════════════════
def load_run(store, run_id, spec):
    rows = store.rows(run_id)
    ok, problems = integrity_check(rows, n_tasks=spec["dataset"]["n"],
                                   n_attempts=spec["budget"]["max_attempts"])
    if not ok:
        raise ValueError(f"run {run_id} 完整性校验失败: {problems}")
    return rows
# 关键：**校验失败就抛异常，不产出报告**。警告会被忽略，异常不会。

# ══════════════════════════════════════════════════════════════════
# B. 常用切片查询（有明细的话都是一句 SQL）
# ══════════════════════════════════════════════════════════════════
# 按难度:
#   SELECT m.difficulty, AVG(r.score), COUNT(*)
#   FROM results r JOIN task_meta m USING (task_id)
#   WHERE r.run_id = ? AND r.status != 'scorer_error'
#   GROUP BY m.difficulty
#
# 只用强判分任务（C66-02 的敏感性分析）:
#   ... AND m.scorer_strength >= 0.6
#
# 首次尝试 vs 重试（看恢复力）:
#   ... GROUP BY r.attempt
#
# 逐题 diff（第 4 节）:
#   SELECT a.task_id, AVG(a.score) sa, AVG(b.score) sb
#   FROM results a JOIN results b USING (task_id)
#   WHERE a.run_id=? AND b.run_id=? GROUP BY a.task_id
#   HAVING (sa<0.5) != (sb<0.5)          -- 只看翻转的题

# ══════════════════════════════════════════════════════════════════
# C. 报告生成的固定顺序（讲解第 7 节）
# ══════════════════════════════════════════════════════════════════
# 1) 完整性校验 → 不通过就停
# 2) 任务集一致性检查 → 不一致就交集重算（练习 2）
# 3) 主指标: micro + macro，**每个都带分母与排除说明**
# 4) 不确定度: 按任务聚类自举（C66-04）
# 5) 与基线 diff: fixed/regressed/churn + McNemar + verdict（练习 3）
# 6) 切片表: 难度 × 子系统
# 7) 成本: $/success（C66-05）
# 8) 运行健康: 五个数（本课模块 02）

# ══════════════════════════════════════════════════════════════════
# D. 保留策略：默认全留
# ══════════════════════════════════════════════════════════════════
# 永远不删: run_id, task_id, attempt, fingerprint, status, score
#           （每行 < 100 字节，百万行也只有几十 MB）
# 可以压缩: output（文本压缩率 5-10 倍）
# 可以分层: 三个月以上归档到对象存储，**但每个 run 的聚合摘要留在热库**
# 差异化:   失败样本的 output 全留，成功样本的可以只留哈希
#           —— 因为你要看的永远是失败的那些
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 宽事实表 | 存最细粒度事实，聚合留到查询时做 | schema 设计 |
| 后置聚合 | 同一批结果算出四个都对但含义不同的数；每个都要带分母 | 报告 |
| 空输入返回 nan | 返回 0 会让「没数据」和「全错」无法区分 | 聚合函数 |
| Simpson 悖论 | 任务集不同则总分不可比——必须交集重算 | 跨 run 比较 |
| run diff | churn / |net| > 4 时该加重复次数，而不是宣布提升 | 判读改动 |
| 五条完整性校验 | 放进**加载函数**，不是放进「记得跑一下」的脚本 | 每次加载 |
| 保留策略 | 百万行几十 GB 是小数据；默认全留，失败样本的 output 尤其不能删 | 存储规划 |

下一模块：**04 · CI 回归门禁**——阈值该怎么从方差推出来、
什么该阻断什么只该警告、以及「误报让团队关掉门禁」这个最终失败模式。